# Vector similarity search and hybrid search

## 1) Imports and functions

In [1]:
import requests
import pdfplumber
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

True

In [2]:
def chunk_text(text, chunk_size, overlap, split_on_whitespace_only=True):
    chunks = []
    index = 0

    while index < len(text):
        if split_on_whitespace_only:
            prev_whitespace = 0
            left_index = index - overlap
            while left_index >= 0:
                if text[left_index] == " ":
                    prev_whitespace = left_index
                    break
                left_index -= 1
            next_whitespace = text.find(" ", index + chunk_size)
            if next_whitespace == -1:
                next_whitespace = len(text)
            chunk = text[prev_whitespace:next_whitespace].strip()
            chunks.append(chunk)
            index = next_whitespace + 1
        else:
            start = max(0, index - overlap + 1)
            end = min(index + chunk_size + overlap, len(text))
            chunk = text[start:end].strip()
            chunks.append(chunk)
            index += chunk_size

    return chunks

In [6]:
# creation of clients
emb_model = OpenAIEmbeddings(
    model=os.getenv("MODEL_EMBEDDING"),
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    check_embedding_ctx_length=False
)

## 2) Load data

In [3]:
remote_pdf_url = "https://arxiv.org/pdf/1709.00666.pdf"
pdf_filename = "./data/ch02-downloaded.pdf"

response = requests.get(remote_pdf_url)

if response.status_code == 200:
    with open(pdf_filename, "wb") as pdf_file:
        pdf_file.write(response.content)
        print("Download concluded!")
else:
    print("Failed to download the PDF. Status code: ", response.status_code)

Download concluded!


In [4]:
text = ""

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

print(f"Extraction successfully completed. Sample: '{text[:50]}...'")

Extraction successfully completed. Sample: 'Einstein’s Patents and Inventions
Asis Kumar Chaud...'


## 3) Chunk creation

In [5]:
chunks = chunk_text(text, 500, 40)
print("Amount of chunks: ", len(chunks))
print("Instance of chunk: ", chunks[0])

Amount of chunks:  89
Instance of chunk:  Einstein’s Patents and Inventions
Asis Kumar Chaudhuri
Variable Energy Cyclotron Centre
1‐AF Bidhan Nagar, Kolkata‐700 064
Abstract: Times magazine selected Albert Einstein, the German born Jewish Scientist as the person of the 20th
century. Undoubtedly, 20th century was the age of science and Einstein’s contributions in unravelling mysteries
of nature was unparalleled. However, few are aware that Einstein was also a great inventor. He and his
collaborators had patented a wide variety of inventions


## 4) Embeddings creation

In [7]:
embeddings = emb_model.embed_documents(chunks)
print("Amount of embeddings: ", len(embeddings))
print("dimension of embeddings: ", len(embeddings[0]))

Amount of embeddings:  89
dimension of embeddings:  384
